In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("Creditcard_data.csv")

print("Original Dataset Shape:")
print(df.shape)

Original Dataset Shape:
(772, 31)


In [6]:
majority = df[df['Class'] == 0]
minority = df[df['Class'] == 1]

minority_upsampled = resample(minority,
                              replace=True,
                              n_samples=len(majority),
                              random_state=42)

In [7]:
balanced_df = pd.concat([majority, minority_upsampled])

print("\nBalanced Dataset Shape:")
print(balanced_df.shape)

print("\nClass Distribution:")
print(balanced_df['Class'].value_counts())


Balanced Dataset Shape:
(1526, 31)

Class Distribution:
Class
0    763
1    763
Name: count, dtype: int64


In [8]:
X = balanced_df.drop('Class', axis=1)
y = balanced_df['Class']

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [11]:
def sampling1_random(X, y):
    return train_test_split(X, y, test_size=0.3, random_state=42)

def sampling2_systematic(X, y, step=2):
    indices = np.arange(0, len(X), step)
    X_sample = X.iloc[indices]
    y_sample = y.iloc[indices]
    return train_test_split(X_sample, y_sample, test_size=0.3, random_state=42)

def sampling3_stratified(X, y):
    return train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

def sampling4_bootstrap(X, y):
    X_boot, y_boot = resample(X, y, replace=True, random_state=42)
    return train_test_split(X_boot, y_boot, test_size=0.3, random_state=42)

def sampling5_balanced_stratified(X, y):
    return train_test_split(X, y, test_size=0.3, stratify=y, random_state=1)

In [12]:
sampling_methods = {
    "Sampling1": sampling1_random,
    "Sampling2": sampling2_systematic,
    "Sampling3": sampling3_stratified,
    "Sampling4": sampling4_bootstrap,
    "Sampling5": sampling5_balanced_stratified
}

In [13]:
models = {
    "M1_LogisticRegression": LogisticRegression(max_iter=1000),
    "M2_DecisionTree": DecisionTreeClassifier(),
    "M3_RandomForest": RandomForestClassifier(),
    "M4_KNN": KNeighborsClassifier(),
    "M5_SVM": SVC()
}

In [14]:
results = pd.DataFrame(index=models.keys(), columns=sampling_methods.keys())

for sampling_name, sampling_function in sampling_methods.items():

    X_train, X_test, y_train, y_test = sampling_function(X, y)

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred) * 100

        results.loc[model_name, sampling_name] = round(accuracy, 2)

In [15]:
print("\nAccuracy Results Table:")
print(results)


Accuracy Results Table:
                      Sampling1 Sampling2 Sampling3 Sampling4 Sampling5
M1_LogisticRegression     90.83     90.83     91.48     93.01     93.67
M2_DecisionTree           100.0     98.25     98.91     99.78     99.13
M3_RandomForest           100.0     100.0     100.0     100.0     100.0
M4_KNN                     97.6     93.45     97.82     97.38     96.94
M5_SVM                     97.6     96.07     97.82     98.03     98.25


In [16]:
print("\nBest Sampling Technique for Each Model:")

best_sampling = {}

for model in results.index:

    best = results.loc[model].astype(float).idxmax()

    best_sampling[model] = best

    print(model, ":", best)


Best Sampling Technique for Each Model:
M1_LogisticRegression : Sampling5
M2_DecisionTree : Sampling1
M3_RandomForest : Sampling1
M4_KNN : Sampling3
M5_SVM : Sampling5


In [17]:
average_accuracy = results.astype(float).mean()

print("\nAverage Accuracy of Each Sampling Technique:")
print(average_accuracy)

overall_best = average_accuracy.idxmax()

print("\nOverall Best Sampling Technique:", overall_best)


Average Accuracy of Each Sampling Technique:
Sampling1    97.206
Sampling2    95.720
Sampling3    97.206
Sampling4    97.640
Sampling5    97.598
dtype: float64

Overall Best Sampling Technique: Sampling4
